In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
import scipy.io.wavfile as wav
!pip install python_speech_features
from python_speech_features import mfcc

# If your code is giving your problems, remove this
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="tensorflow.lite")

In [ ]:
# ============================================
# COLAB SETUP
# ============================================
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print("Google Drive mounted successfully!")
except:
    IN_COLAB = False
    print("Not running in Colab")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted successfully!


In [ ]:
# ============================================
# CONFIGURATION FOR STREAMING DETECTION
# ============================================
SAMPLE_RATE = 16000    # Sample rate of our audio
WINDOW_SIZE_MS = 1250  # 1.25 second window
WINDOW_STRIDE_MS = 20  # 20ms stride (50 inferences/second)
NUM_MFCC = 13          # MFCC parameters
NUM_CLASSES = 3        # Number of categories we train on
CATEGORIES = ['w', 'o', 'n'] # make sure these match your subpaths!

# Paths - you will likely need to update this!
if IN_COLAB:
    DATA_PATH = '/content/drive/MyDrive/wake_word_dataset_taset_oceanpanda667/'
    OUTPUT_PATH = '/content/drive/MyDrive/wake_word_models/'
else:
    DATA_PATH = './data/'
    OUTPUT_PATH = './models/'

os.makedirs(OUTPUT_PATH, exist_ok=True)

In [ ]:
# ============================================
# DATA AUGMENTATION
# ============================================

def add_noise(signal, noise_factor=0.005):
    noise = np.random.randn(len(signal))
    return signal + noise_factor * noise

def time_shift(signal, shift_max=0.5):
    """Aggressive time shift for position invariance"""
    shift = np.random.randint(-int(shift_max * len(signal)),
                               int(shift_max * len(signal)))
    augmented = np.roll(signal, shift)
    if shift > 0:
        augmented[:shift] = 0
    else:
        augmented[shift:] = 0
    return augmented

def change_volume(signal, volume_factor=None):
    if volume_factor is None:
        volume_factor = np.random.uniform(0.5, 1.5)
    return signal * volume_factor

def time_stretch(signal, rate=None):
    if rate is None:
        rate = np.random.uniform(0.8, 1.2)
    indices = np.arange(0, len(signal), rate)
    indices = indices[indices < len(signal)].astype(int)
    stretched = signal[indices]
    if len(stretched) < len(signal):
        stretched = np.pad(stretched, (0, len(signal) - len(stretched)), 'constant')
    else:
        stretched = stretched[:len(signal)]
    return stretched

def add_context_audio(signal, context_type='random'):
    """Add noise before/after the word to *roughly* simulate other conversation"""
    window_length = len(signal)
    threshold = np.max(np.abs(signal)) * 0.1
    voice_indices = np.where(np.abs(signal) > threshold)[0]

    if len(voice_indices) == 0:
        return signal

    start_voice = voice_indices[0]
    end_voice = voice_indices[-1]

    if context_type == 'random' or context_type == 'before':
        if start_voice > 1000:
            noise_length = np.random.randint(500, start_voice)
            noise = np.random.randn(noise_length) * 0.05
            signal[:noise_length] = noise

    if context_type == 'random' or context_type == 'after':
        if end_voice < window_length - 1000:
            noise_start = end_voice + 1
            noise_length = np.random.randint(500, window_length - noise_start)
            noise = np.random.randn(noise_length) * 0.05
            signal[noise_start:noise_start+noise_length] = noise

    return signal

def random_position_in_window(signal):
    """Place word at random position in window (simulates streaming)"""
    threshold = np.max(np.abs(signal)) * 0.1
    voice_indices = np.where(np.abs(signal) > threshold)[0]

    if len(voice_indices) == 0:
        return signal

    start_voice = voice_indices[0]
    end_voice = voice_indices[-1]
    voice_segment = signal[start_voice:end_voice+1]

    window_length = len(signal)
    voice_length = len(voice_segment)

    if voice_length >= window_length:
        return signal

    max_start = window_length - voice_length
    new_start = np.random.randint(0, max_start + 1)

    new_signal = np.zeros(window_length, dtype=signal.dtype)
    new_signal[new_start:new_start+voice_length] = voice_segment

    if np.random.random() > 0.5:
        new_signal = add_context_audio(new_signal,
                                       context_type=np.random.choice(['before', 'after', 'random']))
    return new_signal

def augment_audio(signal):
    """Apply random augmentation"""
    aug_type = np.random.choice(['noise', 'time_shift', 'volume', 'stretch',
                                 'position', 'combined'])

    if aug_type == 'noise':
        return add_noise(signal, np.random.uniform(0.001, 0.01))
    elif aug_type == 'time_shift':
        return time_shift(signal, 0.5)
    elif aug_type == 'volume':
        return change_volume(signal, np.random.uniform(0.6, 1.4))
    elif aug_type == 'stretch':
        return time_stretch(signal, np.random.uniform(0.85, 1.15))
    elif aug_type == 'position':
        return random_position_in_window(signal)
    elif aug_type == 'combined':
        aug_signal = signal.copy()
        for _ in range(np.random.randint(2, 4)):
            aug_signal = augment_audio(aug_signal)
        return aug_signal
    return signal


In [ ]:
# ============================================
# DATA LOADING
# ============================================
def load_dataset(augment=True, aug_factor=5):
    """Load audio files and extract MFCC features"""
    X, y = [], []
    window_samples = int(SAMPLE_RATE * WINDOW_SIZE_MS / 1000)

    for label_idx, category in enumerate(CATEGORIES):
        path = os.path.join(DATA_PATH, category)
        # This takes a max of 100 files - just the first 100 it finds.
        files = [f for f in os.listdir(path) if f.endswith('.wav')][:100]

        print(f"Loading {category}: {len(files)} files")

        for file in files:
            # Load audio
            rate, signal = wav.read(os.path.join(path, file))
            signal = signal.astype(np.float32) / 32768.0

            # Pad or truncate
            if len(signal) < window_samples:
                signal = np.pad(signal, (0, window_samples - len(signal)))
            else:
                signal = signal[:window_samples]

            # Extract features
            features = mfcc(signal, samplerate=SAMPLE_RATE, numcep=NUM_MFCC)
            X.append(features)
            y.append(label_idx)

            # Augment
            if augment:
                for _ in range(aug_factor - 1):
                    aug_signal = augment_audio(signal)
                    aug_features = mfcc(aug_signal, samplerate=SAMPLE_RATE, numcep=NUM_MFCC)
                    X.append(aug_features)
                    y.append(label_idx)

    return np.array(X)[..., np.newaxis], np.array(y)

In [ ]:
# ============================================
# MODEL
# ============================================
def create_model(input_shape, num_classes):
    """Create lightweight CNN for wake word detection"""
    return keras.Sequential([
        keras.layers.Input(shape=input_shape),
        keras.layers.SeparableConv2D(32, (3, 3), activation='relu', padding='same'),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Dropout(0.2),

        keras.layers.SeparableConv2D(64, (3, 3), activation='relu', padding='same'),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Dropout(0.2),

        keras.layers.GlobalAveragePooling2D(),
        keras.layers.Dense(32, activation='relu'),
        keras.layers.Dense(num_classes, activation='softmax')
    ])

In [ ]:
# ============================================
# TRAINING
# ============================================
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    mode='min',
    patience=76,
    restore_best_weights=True,
    verbose=1 # Optional: displays a message when stopping
)

def train_model(augment=True):
    """Train wake word model"""
    print("Loading data...")
    X, y = load_dataset(augment, aug_factor=5)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

    print(f"\nTrain: {len(X_train)}, Test: {len(X_test)}")
    print(f"Input shape: {X_train[0].shape}\n")

    model = create_model(X_train[0].shape, len(CATEGORIES))
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    model.summary()

    print("\nTraining...")
    model.fit(X_train, y_train, validation_split=0.2, epochs=80, batch_size=32,
              callbacks=[early_stopping])

    test_acc = model.evaluate(X_test, y_test, verbose=0)[1]
    print(f"\nTest accuracy: {test_acc*100:.1f}%")

    model.save(os.path.join(OUTPUT_PATH, 'wake_word_model.h5'))
    return model, X_test, y_test

In [ ]:
# ============================================
# TFLITE CONVERSION
# ============================================
def convert_to_tflite(model, X_test, quantize=True):
    """Convert to TFLite (optionally quantized)"""
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    if quantize:
        def representative_dataset():
            for i in range(min(100, len(X_test))):
                yield [X_test[i:i+1].astype(np.float32)]

        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = representative_dataset
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8

    tflite_model = converter.convert()
    suffix = 'quantized' if quantize else 'float32'
    output_path = os.path.join(OUTPUT_PATH, f'wake_word_{suffix}.tflite')

    with open(output_path, 'wb') as f:
        f.write(tflite_model)

    print(f"TFLite model saved: {output_path} ({len(tflite_model)/1024:.1f} KB)")
    return output_path

In [ ]:
def test_tflite(tflite_path, X_test, y_test):
    """Test TFLite model accuracy and speed"""
    import time
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    input_scale, input_zero_point = input_details[0]['quantization']

    correct = 0
    times = []

    for i in range(len(X_test)):
        x = X_test[i:i+1]
        if input_scale > 0:  # Quantized model
            x = x / input_scale + input_zero_point
            x = x.astype(input_details[0]['dtype'])
        else: # float32 model
            x = x.astype(np.float32)

        start = time.time()
        interpreter.set_tensor(input_details[0]['index'], x)
        interpreter.invoke()
        times.append((time.time() - start) * 1000)

        output = interpreter.get_tensor(output_details[0]['index'])
        if np.argmax(output) == y_test[i]:
            correct += 1

    print(f"Accuracy: {correct/len(X_test)*100:.1f}%")
    print(f"Avg inference: {np.mean(times):.2f}ms")

In [ ]:
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Set augment to True/False to toggle data augmentation
model, X_test, y_test = train_model(augment=True)



Loading data...
Loading w: 100 files
Loading o: 100 files
Loading n: 100 files

Train: 1200, Test: 300
Input shape: (124, 13, 1)



Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ separable_conv2d_12             │ (None, 124, 13, 32)    │            73 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_12 (MaxPooling2D) │ (None, 62, 6, 32)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 62, 6, 32)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d_13             │ (None, 62, 6, 64)      │         2,400 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_13 (MaxPooling2D) │ (None, 31, 3, 64)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 31, 3, 64)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_6      │ (None, 64)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,652 (18.17 KB)

 Trainable params: 4,652 (18.17 KB)

 Non-trainable params: 0 (0.00 B)


Training...
Epoch 1/80
30/30 ━━━━━━━━━━━━━━━━━━━━ 5s 75ms/step - accuracy: 0.3260 - loss: 1.1138 - val_accuracy: 0.3292 - val_loss: 1.0932
Epoch 2/80
30/30 ━━━━━━━━━━━━━━━━━━━━ 2s 67ms/step - accuracy: 0.3656 - loss: 1.0938 - val_accuracy: 0.3417 - val_loss: 1.0834
Epoch 3/80
30/30 ━━━━━━━━━━━━━━━━━━━━ 4s 114ms/step - accuracy: 0.3521 - loss: 1.0846 - val_accuracy: 0.5125 - val_loss: 1.0804
Epoch 4/80
30/30 ━━━━━━━━━━━━━━━━━━━━ 2s 74ms/step - accuracy: 0.4302 - loss: 1.0751 - val_accuracy: 0.4875 - val_loss: 1.0590
Epoch 5/80
30/30 ━━━━━━━━━━━━━━━━━━━━ 2s 65ms/step - accuracy: 0.4917 - loss: 1.0562 - val_accuracy: 0.4458 - val_loss: 1.0536
Epoch 6/80
30/30 ━━━━━━━━━━━━━━━━━━━━ 2s 63ms/step - accuracy: 0.5104 - loss: 1.0309 - val_accuracy: 0.5542 - val_loss: 1.0189
Epoch 7/80
30/30 ━━━━━━━━━━━━━━━━━━━━ 2s 64ms/step - accuracy: 0.5604 - loss: 0.9870 - val_accuracy: 0.5833 - val_loss: 0.9637
Epoch 8/80
30/30 ━━━━━━━━━━━━━━━━━━━━ 3s 66ms/step - accuracy: 0.5740 - loss: 0.9443 - val_accura


Test accuracy: 86.7%


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Convert to TF Lite - both quantized(8bit) and regular (32bit)
quantized_path = convert_to_tflite(model, X_test, quantize=True)
float_path = convert_to_tflite(model, X_test, quantize=False)

Saved artifact at '/tmp/tmpnmaof_oc'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 124, 13, 1), dtype=tf.float32, name='keras_tensor_60')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  133262230197392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133262230194512: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133261694510928: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133262230195472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133262234771216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133262237088848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133262237083472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133262230196048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133261694507856: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133262237081744: TensorSpec(shape=(), dtype=tf.resource, name=None)
TFLite model s

In [ ]:
# Convert both quantized versions
print("\nTesting quantized:")
test_tflite(quantized_path, X_test, y_test)
print("\nTesting float32:")
test_tflite(float_path, X_test, y_test)


Testing quantized:
Accuracy: 86.3%
Avg inference: 0.19ms

Testing float32:
Accuracy: 86.7%
Avg inference: 0.11ms


In [ ]:
# --- Week 1 Task 3.1 Model Evaluation Code Snippet ---
import numpy as np

def calculate_regression_metrics(y_true, y_pred):
    # Calculate raw residuals
    errors = y_true - y_pred

    # Calculate final required evaluation metrics
    mae = np.mean(np.abs(errors))
    rmse = np.sqrt(np.mean(errors ** 2))
    max_error = np.max(np.abs(errors))

    print(f"RMSE: {rmse:.6f} | MAE: {mae:.6f} | Max Error: {max_error:.6f}")
    return rmse, mae, max_error